# SE-ResNet50 + 2-scale Bilinear + SAM + SWA — v3

Différences vs v1 et v2 :
- **2-scale bilinear** : layer3+layer4 seulement (vs v2 qui avait layer2+layer3+layer4 → layer2 ajoutait du bruit)
- **SAM + SWA** corrigé (vs v1 qui n'avait ni l'un ni l'autre)
- **Seed 123** : folds différents → plus de diversité pour l'ensemble
- **Rotation 90°** + GaussianBlur : augmentations plus adaptées aux textures
- **Patch only** : 300 epochs, patience 60
- Dest ensemble : v1_patch + v2_patch + v2_full + v3_patch → 4 modèles

## 1. Setup

In [1]:
from pathlib import Path
import random, time, math

import numpy as np
import pandas as pd
from PIL import Image

from sklearn.model_selection import StratifiedKFold

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.optim.swa_utils import AveragedModel
from torchvision import transforms
from tqdm.auto import tqdm

ROOT           = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
DATA_DIR       = ROOT / 'data'
OUTPUT_DIR     = ROOT / 'outputs'
CHECKPOINT_DIR = OUTPUT_DIR / 'checkpoints'
SUBMISSION_DIR = OUTPUT_DIR / 'submissions'

for p in [CHECKPOINT_DIR, SUBMISSION_DIR]:
    p.mkdir(parents=True, exist_ok=True)

SEED = 123
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.benchmark = True

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Root  :', ROOT)
print('Device:', DEVICE)
if torch.cuda.is_available():
    print('GPU   :', torch.cuda.get_device_name(0))
    print(f'VRAM  : {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB')


Root  : /home/onyxia/work/tilda-texture-classification
Device: cuda
GPU   : NVIDIA RTXA6000-48Q
VRAM  : 51.3 GB


## 2. Paramètres

In [2]:
N_SPLITS = 5
START_FOLD_PATCH = 1

NUM_CLASSES = 8
NUM_WORKERS = 0

PATCH_RESIZE  = (512, 768)
PATCH_SIZE    = 384

BATCH_SIZE_PATCH = 32

EPOCHS_PATCH  = 300
WARMUP_EPOCHS = 10

LR_PATCH        = 0.05
MOMENTUM        = 0.9
WEIGHT_DECAY    = 1e-3
LABEL_SMOOTHING = 0.1
PATIENCE        = 60

BILINEAR_REDUCTION = 64

MIXUP_ALPHA  = 0.4
CUTMIX_ALPHA = 1.0
CUTMIX_PROB  = 0.5

SWA_START_RATIO = 0.65
SAM_RHO         = 0.05

print('Config OK')
R = BILINEAR_REDUCTION
print(f'Features/echelle : {R*(R+1)//2}  |  Total FC input (2 echelles) : {2*R*(R+1)//2}')


Config OK
Features/echelle : 2080  |  Total FC input (2 echelles) : 4160


## 3. Données

In [3]:
def read_train_csv(path):
    try:
        df = pd.read_csv(path, sep=';')
        if len(df.columns) == 1:
            df = pd.read_csv(path)
    except Exception:
        df = pd.read_csv(path)
    df.columns = [c.strip() for c in df.columns]
    if 'id' not in df.columns or 'label' not in df.columns:
        raise ValueError(f'Colonnes attendues id,label. Trouvees: {df.columns.tolist()}')
    df['id'] = df['id'].astype(int)
    df['label'] = df['label'].astype(int)
    return df

def resolve_image_path(folder, image_id):
    stem = str(int(image_id))
    for ext in ['.tif', '.tiff', '.png', '.jpg']:
        p = folder / f'{stem}{ext}'
        if p.exists(): return p
    matches = sorted(folder.glob(f'{stem}.*'))
    if matches: return matches[0]
    raise FileNotFoundError(f'Image introuvable id={image_id}')

train_csv = DATA_DIR / 'train.csv'
train_dir = DATA_DIR / 'train'
test_dir  = DATA_DIR / 'test'

df = read_train_csv(train_csv)
df['path'] = df['id'].map(lambda x: resolve_image_path(train_dir, x))

test_paths = sorted(test_dir.glob('*.*'), key=lambda p: int(p.stem))
test_df = pd.DataFrame({'id': [int(p.stem) for p in test_paths], 'path': test_paths})

print(df.head())
print('\nDistribution:', df['label'].value_counts().sort_index().to_dict())
print(f'Train: {len(df)}  Test: {len(test_df)}')


   id  label                                               path
0   1      4  /home/onyxia/work/tilda-texture-classification...
1   2      6  /home/onyxia/work/tilda-texture-classification...
2   3      7  /home/onyxia/work/tilda-texture-classification...
3   4      6  /home/onyxia/work/tilda-texture-classification...
4   5      7  /home/onyxia/work/tilda-texture-classification...

Distribution: {0: 300, 1: 300, 2: 262, 3: 300, 4: 300, 5: 299, 6: 300, 7: 300}
Train: 2361  Test: 789


## 4. Transforms + Dataset

Augmentations clés :
- `RandomRotation(90)` : les textures sont invariantes à la rotation
- `GaussianBlur` : robustesse au flou → meilleure généralisation

In [4]:
patch_train_tfms = transforms.Compose([
    transforms.Resize(PATCH_RESIZE),
    transforms.RandomCrop((PATCH_SIZE, PATCH_SIZE)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomVerticalFlip(p=0.5),
    transforms.RandomRotation(degrees=90),
    transforms.GaussianBlur(kernel_size=5, sigma=(0.1, 1.5)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5], std=[0.5]),
    transforms.RandomErasing(p=0.10, scale=(0.02, 0.06), ratio=(0.3, 3.3)),
])

patch_eval_full_tfms = transforms.Compose([
    transforms.Resize(PATCH_RESIZE),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5], std=[0.5]),
])


class TildaDataset(Dataset):
    def __init__(self, frame, transform=None, has_labels=True):
        self.frame = frame.reset_index(drop=True).copy()
        self.transform = transform
        self.has_labels = has_labels

    def __len__(self): return len(self.frame)

    def __getitem__(self, idx):
        row = self.frame.iloc[idx]
        image = Image.open(row['path']).convert('L')
        if self.transform is not None:
            image = self.transform(image)
        label = int(row['label']) if self.has_labels else -1
        return image, label, int(row['id'])


def make_loader(frame, transform, batch_size, shuffle=False, has_labels=True):
    ds = TildaDataset(frame, transform=transform, has_labels=has_labels)
    return DataLoader(ds, batch_size=batch_size, shuffle=shuffle,
                      num_workers=NUM_WORKERS, pin_memory=torch.cuda.is_available())


## 5. MixUp / CutMix

In [5]:
def rand_bbox(h, w, lam):
    cut_h = int(h * np.sqrt(1.0 - lam))
    cut_w = int(w * np.sqrt(1.0 - lam))
    cx, cy = np.random.randint(w), np.random.randint(h)
    x1, x2 = max(cx - cut_w // 2, 0), min(cx + cut_w // 2, w)
    y1, y2 = max(cy - cut_h // 2, 0), min(cy + cut_h // 2, h)
    return x1, y1, x2, y2


def mixup_cutmix(images, labels):
    B, C, H, W = images.shape
    idx = torch.randperm(B, device=images.device)
    if random.random() < CUTMIX_PROB:
        lam = float(np.random.beta(CUTMIX_ALPHA, CUTMIX_ALPHA))
        x1, y1, x2, y2 = rand_bbox(H, W, lam)
        images = images.clone()
        images[:, :, y1:y2, x1:x2] = images[idx, :, y1:y2, x1:x2]
        lam = 1.0 - (x2 - x1) * (y2 - y1) / (H * W)
    else:
        lam = float(np.random.beta(MIXUP_ALPHA, MIXUP_ALPHA))
        images = lam * images + (1.0 - lam) * images[idx]
    return images, labels, labels[idx], lam

print('MixUp/CutMix OK')


MixUp/CutMix OK


## 6. SAM Optimizer

In [6]:
class SAM(torch.optim.Optimizer):
    def __init__(self, params, base_optimizer, rho=0.05, **kwargs):
        assert rho >= 0.0
        defaults = dict(rho=rho, **kwargs)
        super().__init__(params, defaults)
        self.base_optimizer = base_optimizer(self.param_groups, **kwargs)
        self.param_groups   = self.base_optimizer.param_groups
        self.defaults.update(self.base_optimizer.defaults)

    @torch.no_grad()
    def first_step(self, zero_grad=False):
        grad_norm = self._grad_norm()
        for group in self.param_groups:
            scale = group['rho'] / (grad_norm + 1e-12)
            for p in group['params']:
                if p.grad is None: continue
                self.state[p]['old_p'] = p.data.clone()
                p.add_(p.grad * scale.to(p))
        if zero_grad: self.zero_grad()

    @torch.no_grad()
    def second_step(self, zero_grad=False):
        for group in self.param_groups:
            for p in group['params']:
                if p.grad is None: continue
                p.data = self.state[p]['old_p']
        self.base_optimizer.step()
        if zero_grad: self.zero_grad()

    def _grad_norm(self):
        dev = self.param_groups[0]['params'][0].device
        return torch.norm(torch.stack([
            p.grad.norm(p=2).to(dev)
            for group in self.param_groups
            for p in group['params'] if p.grad is not None
        ]), p=2)

    def load_state_dict(self, state_dict):
        super().load_state_dict(state_dict)
        self.base_optimizer.param_groups = self.param_groups

print('SAM OK')


SAM OK


## 7. Architecture — SE-ResNet50 + 2-scale Bilinear

- layer3 (1024ch) → bpool3 → 2080 features
- layer4 (2048ch) → bpool4 → 2080 features
- Concat → **4160 features** → Dropout(0.5) → FC(4160, 8)

Sans layer2 : features de plus haut niveau → meilleure généralisation

In [7]:
class SEBlock(nn.Module):
    def __init__(self, channels, reduction=16):
        super().__init__()
        hidden = max(channels // reduction, 4)
        self.pool = nn.AdaptiveAvgPool2d(1)
        self.fc = nn.Sequential(
            nn.Linear(channels, hidden),
            nn.ReLU(inplace=True),
            nn.Linear(hidden, channels),
            nn.Sigmoid(),
        )

    def forward(self, x):
        b, c = x.shape[:2]
        w = self.pool(x).view(b, c)
        return x * self.fc(w).view(b, c, 1, 1)


class SEBottleneck(nn.Module):
    expansion = 4

    def __init__(self, in_planes, planes, stride=1, se_reduction=16):
        super().__init__()
        self.conv1 = nn.Conv2d(in_planes, planes, 1, bias=False)
        self.bn1   = nn.BatchNorm2d(planes)
        self.conv2 = nn.Conv2d(planes, planes, 3, stride=stride, padding=1, bias=False)
        self.bn2   = nn.BatchNorm2d(planes)
        self.conv3 = nn.Conv2d(planes, planes * 4, 1, bias=False)
        self.bn3   = nn.BatchNorm2d(planes * 4)
        self.se    = SEBlock(planes * 4, reduction=se_reduction)
        self.shortcut = nn.Sequential()
        if stride != 1 or in_planes != planes * 4:
            self.shortcut = nn.Sequential(
                nn.Conv2d(in_planes, planes * 4, 1, stride=stride, bias=False),
                nn.BatchNorm2d(planes * 4),
            )

    def forward(self, x):
        out = F.relu(self.bn1(self.conv1(x)), inplace=True)
        out = F.relu(self.bn2(self.conv2(out)), inplace=True)
        out = self.se(self.bn3(self.conv3(out)))
        return F.relu(out + self.shortcut(x), inplace=True)


class BilinearPool(nn.Module):
    def __init__(self, in_channels, reduction=64):
        super().__init__()
        R = reduction
        self.reduce = nn.Sequential(
            nn.Conv2d(in_channels, R, 1, bias=False),
            nn.BatchNorm2d(R),
            nn.ReLU(inplace=True),
        )
        rows, cols = torch.triu_indices(R, R)
        self.register_buffer('triu_r', rows)
        self.register_buffer('triu_c', cols)
        self.out_features = R * (R + 1) // 2

    def forward(self, x):
        x = self.reduce(x)
        B, R, H, W = x.shape
        x = x.view(B, R, H * W)
        cov = torch.bmm(x, x.transpose(1, 2)) / (H * W)
        feat = cov[:, self.triu_r, self.triu_c]
        feat = torch.sign(feat) * torch.sqrt(torch.abs(feat) + 1e-10)
        return F.normalize(feat, dim=1)


class SEResNet50Bilinear2Scale(nn.Module):
    def __init__(self, num_classes=8, in_channels=1, reduction=64):
        super().__init__()
        self._in = 64
        self.stem = nn.Sequential(
            nn.Conv2d(in_channels, 64, 7, stride=2, padding=3, bias=False),
            nn.BatchNorm2d(64), nn.ReLU(inplace=True),
            nn.MaxPool2d(3, stride=2, padding=1),
        )
        self.layer1 = self._make(64,  3, stride=1)
        self.layer2 = self._make(128, 4, stride=2)
        self.layer3 = self._make(256, 6, stride=2)
        self.layer4 = self._make(512, 3, stride=2)

        self.bpool3 = BilinearPool(1024, reduction=reduction)
        self.bpool4 = BilinearPool(2048, reduction=reduction)

        feat_dim = 2 * reduction * (reduction + 1) // 2
        self.dropout = nn.Dropout(p=0.5)
        self.fc = nn.Linear(feat_dim, num_classes)

        self.apply(self._init_w)
        nn.init.xavier_normal_(self.fc.weight, gain=0.01)
        nn.init.zeros_(self.fc.bias)

    def _make(self, planes, n, stride=1):
        layers = []
        for s in [stride] + [1] * (n - 1):
            layers.append(SEBottleneck(self._in, planes, stride=s))
            self._in = planes * 4
        return nn.Sequential(*layers)

    def _init_w(self, m):
        if isinstance(m, nn.Conv2d):
            nn.init.kaiming_normal_(m.weight, mode='fan_out', nonlinearity='relu')
        elif isinstance(m, nn.BatchNorm2d):
            nn.init.ones_(m.weight); nn.init.zeros_(m.bias)
        elif isinstance(m, nn.Linear):
            nn.init.xavier_normal_(m.weight)
            if m.bias is not None: nn.init.zeros_(m.bias)

    def forward(self, x):
        x  = self.stem(x)
        x  = self.layer1(x)
        x  = self.layer2(x)
        x3 = self.layer3(x)
        x4 = self.layer4(x3)
        f3 = self.bpool3(x3)
        f4 = self.bpool4(x4)
        return self.fc(self.dropout(torch.cat([f3, f4], dim=1)))


def seresnet50_bilinear_v3(num_classes=8):
    return SEResNet50Bilinear2Scale(num_classes=num_classes, in_channels=1,
                                    reduction=BILINEAR_REDUCTION)


m = seresnet50_bilinear_v3()
n_params = sum(p.numel() for p in m.parameters())
print(f'SE-ResNet50-Bilinear-2scale-v3 params: {n_params/1e6:.2f}M')
print(f'Feature dim FC: {2 * BILINEAR_REDUCTION * (BILINEAR_REDUCTION+1) // 2}')
del m


SE-ResNet50-Bilinear-2scale-v3 params: 26.26M
Feature dim FC: 4160


## 8. Eval + TTA 36 variants

In [8]:
@torch.no_grad()
def evaluate(model, loader, criterion):
    model.eval()
    total_loss, correct, n = 0.0, 0, 0
    y_true, y_pred = [], []
    for images, labels, _ in tqdm(loader, leave=False):
        images = images.to(DEVICE, non_blocking=True)
        labels = labels.to(DEVICE, non_blocking=True)
        logits = model(images)
        loss = criterion(logits, labels)
        total_loss += loss.item() * labels.size(0)
        preds = logits.argmax(1)
        correct += (preds == labels).sum().item()
        n += labels.size(0)
        y_true.extend(labels.cpu().numpy().tolist())
        y_pred.extend(preds.cpu().numpy().tolist())
    return correct / n, total_loss / n, np.array(y_true), np.array(y_pred)


def crop_grid(images, crop_size=384):
    _, _, h, w = images.shape
    top  = [0, max((h - crop_size) // 2, 0), max(h - crop_size, 0)]
    left = [0, max((w - crop_size) // 2, 0), max(w - crop_size, 0)]
    return [images[:, :, t:t+crop_size, l:l+crop_size]
            for t in top for l in left]


@torch.no_grad()
def predict_patch_tta36(model, loader, crop_size=384):
    model.eval()
    all_probs, all_ids = [], []
    for images, _, image_ids in tqdm(loader, leave=False):
        images = images.to(DEVICE, non_blocking=True)
        prob_list = []
        for crop in crop_grid(images, crop_size=crop_size):
            for v in [crop,
                      torch.flip(crop, dims=[-1]),
                      torch.flip(crop, dims=[-2]),
                      torch.flip(crop, dims=[-2, -1])]:
                prob_list.append(torch.softmax(model(v), dim=1))
        probs = torch.stack(prob_list).mean(0)
        all_probs.append(probs.cpu())
        all_ids.extend(image_ids.numpy().tolist())
    return torch.cat(all_probs).numpy(), np.array(all_ids)


def save_submission(ids, probs, path):
    sub = pd.DataFrame({'id': ids, 'label': probs.argmax(1).astype(int)}).sort_values('id')
    sub.to_csv(path, index=False)
    print('Saved:', path)
    print(sub['label'].value_counts().sort_index())
    return sub

print('Inference OK')


Inference OK


## 9. Entraînement SAM + SWA

In [9]:
def make_scheduler(optimizer, warmup_epochs, total_epochs, base_lr):
    min_lr = base_lr * 0.01
    def fn(epoch):
        if epoch < warmup_epochs:
            return (epoch + 1) / warmup_epochs
        progress = (epoch - warmup_epochs) / max(total_epochs - warmup_epochs, 1)
        cosine = 0.5 * (1.0 + math.cos(math.pi * progress))
        return min_lr / base_lr + (1.0 - min_lr / base_lr) * cosine
    return torch.optim.lr_scheduler.LambdaLR(optimizer, fn)


def train_one_epoch_sam(model, loader, criterion, optimizer):
    model.train()
    total_loss, correct, n = 0.0, 0, 0
    for images, labels, _ in tqdm(loader, leave=False):
        images = images.to(DEVICE, non_blocking=True)
        labels = labels.to(DEVICE, non_blocking=True)
        images_m, la, lb, lam = mixup_cutmix(images, labels)

        logits = model(images_m)
        loss   = lam * criterion(logits, la) + (1.0 - lam) * criterion(logits, lb)
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), 5.0)
        optimizer.first_step(zero_grad=True)

        logits2 = model(images_m)
        loss2   = lam * criterion(logits2, la) + (1.0 - lam) * criterion(logits2, lb)
        loss2.backward()
        nn.utils.clip_grad_norm_(model.parameters(), 5.0)
        optimizer.second_step(zero_grad=True)

        total_loss += loss.item() * labels.size(0)
        correct    += (logits.detach().argmax(1) == la).sum().item()
        n          += labels.size(0)
    return correct / n, total_loss / n


def update_bn_swa(swa_model, loader):
    swa_model.train()
    for m in swa_model.modules():
        if isinstance(m, nn.BatchNorm2d):
            m.reset_running_stats()
            m.momentum = None
    with torch.no_grad():
        for images, _, _ in tqdm(loader, desc='SWA BN', leave=False):
            swa_model(images.to(DEVICE))
    for m in swa_model.modules():
        if isinstance(m, nn.BatchNorm2d):
            m.momentum = 0.1
    swa_model.eval()


def fit_fold(model, train_loader, val_loader, run_name, epochs, lr):
    criterion = nn.CrossEntropyLoss(label_smoothing=LABEL_SMOOTHING)
    optimizer = SAM(model.parameters(), torch.optim.SGD, rho=SAM_RHO,
                    lr=lr, momentum=MOMENTUM, weight_decay=WEIGHT_DECAY, nesterov=True)
    scheduler = make_scheduler(optimizer.base_optimizer, WARMUP_EPOCHS, epochs, lr)

    swa_model = AveragedModel(model)
    swa_start = max(int(SWA_START_RATIO * epochs), 30)
    swa_n     = 0

    best_acc   = -1.0
    best_epoch = 0
    best_path  = CHECKPOINT_DIR / f'best_{run_name}.pt'
    history    = []
    t0         = time.time()

    for epoch in range(1, epochs + 1):
        train_acc, train_loss = train_one_epoch_sam(model, train_loader, criterion, optimizer)
        val_acc, val_loss, _, _ = evaluate(model, val_loader, criterion)
        scheduler.step()

        if epoch >= swa_start:
            swa_model.update_parameters(model)
            swa_n += 1

        if val_acc > best_acc:
            best_acc, best_epoch = val_acc, epoch
            torch.save({'epoch': epoch, 'model_state_dict': model.state_dict(),
                        'best_acc': best_acc}, best_path)

        history.append({'epoch': epoch, 'train_acc': train_acc, 'train_loss': train_loss,
                        'val_acc': val_acc, 'val_loss': val_loss,
                        'lr': scheduler.get_last_lr()[0]})

        print(f'{run_name} | ep {epoch:03d} | train {train_acc:.4f}/{train_loss:.4f} | '
              f'val {val_acc:.4f}/{val_loss:.4f} | best {best_acc:.4f} @ {best_epoch}')

        if epoch - best_epoch >= PATIENCE:
            print(f'Early stop: {PATIENCE} epochs sans amelioration')
            break

    if swa_n >= 5:
        update_bn_swa(swa_model, train_loader)
        swa_val_acc, _, _, _ = evaluate(swa_model, val_loader, criterion)
        print(f'SWA ({swa_n} snapshots) val acc: {swa_val_acc:.4f}  |  best ckpt: {best_acc:.4f}')
        if swa_val_acc >= best_acc:
            best_acc = swa_val_acc
            swa_sd = {k[len('module.'):]: v
                      for k, v in swa_model.state_dict().items()
                      if k.startswith('module.')}
            torch.save({'epoch': epoch, 'model_state_dict': swa_sd,
                        'best_acc': best_acc, 'is_swa': True}, best_path)
            print('-> SWA model saved as best')
        else:
            ckpt = torch.load(best_path, map_location=DEVICE, weights_only=True)
            model.load_state_dict(ckpt['model_state_dict'])
            print('-> Keeping best checkpoint')

    elapsed = (time.time() - t0) / 60
    return pd.DataFrame(history), best_path, best_acc, best_epoch, elapsed

print('fit_fold OK')


fit_fold OK


## 10. Boucle 5-fold patch

In [10]:
def run_5fold_patch(experiment_name, model_factory, start_fold=1):
    skf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=SEED)
    fold_probs   = []
    ids_ref      = None
    fold_results = []

    test_loader = make_loader(test_df, patch_eval_full_tfms, batch_size=BATCH_SIZE_PATCH,
                              shuffle=False, has_labels=False)

    for fold, (tr_idx, va_idx) in enumerate(skf.split(df, df['label']), start=1):
        fold_name  = f'{experiment_name}_fold{fold}'
        probs_path = CHECKPOINT_DIR / f'probs_{fold_name}.npy'
        ids_path   = CHECKPOINT_DIR / f'ids_{fold_name}.npy'

        if fold < start_fold:
            if probs_path.exists() and ids_path.exists():
                probs = np.load(probs_path)
                ids   = np.load(ids_path)
                fold_probs.append(probs)
                ids_ref = ids if ids_ref is None else ids_ref
                print(f'Fold {fold}: loaded saved probs {probs.shape}')
            else:
                print(f'Fold {fold}: skipped (no saved probs found)')
            continue

        print('\n' + '='*80)
        print(f'{experiment_name} | fold {fold}/{N_SPLITS}')
        print('='*80)

        tr_df = df.iloc[tr_idx].reset_index(drop=True)
        va_df = df.iloc[va_idx].reset_index(drop=True)
        train_loader = make_loader(tr_df, patch_train_tfms, BATCH_SIZE_PATCH, shuffle=True)
        val_loader   = make_loader(va_df, patch_eval_full_tfms, BATCH_SIZE_PATCH, shuffle=False)

        model = model_factory().to(DEVICE)
        hist_df, best_path, best_acc, best_epoch, elapsed = \
            fit_fold(model, train_loader, val_loader, fold_name, EPOCHS_PATCH, LR_PATCH)
        hist_df.to_csv(OUTPUT_DIR / f'history_{fold_name}.csv', index=False)

        ckpt = torch.load(best_path, map_location=DEVICE, weights_only=True)
        model.load_state_dict(ckpt['model_state_dict'])

        probs, ids = predict_patch_tta36(model, test_loader, crop_size=PATCH_SIZE)
        np.save(probs_path, probs)
        np.save(ids_path,   ids)
        if ids_ref is None:
            ids_ref = ids
        else:
            assert np.array_equal(ids_ref, ids), 'Test ids differents entre folds!'

        fold_probs.append(probs)
        save_submission(ids, probs,
                        SUBMISSION_DIR / f'submission_{fold_name}_tta_labels0.csv')

        fold_results.append({
            'fold': fold, 'best_val_acc': best_acc,
            'best_epoch': best_epoch, 'training_time_min': elapsed,
        })

        del model
        torch.cuda.empty_cache()

    results_df = pd.DataFrame(fold_results)
    results_df.to_csv(OUTPUT_DIR / f'results_{experiment_name}.csv', index=False)

    if len(fold_probs) == N_SPLITS:
        mean_probs = np.mean(fold_probs, axis=0)
        np.save(CHECKPOINT_DIR / f'probs_{experiment_name}_5fold.npy', mean_probs)
        np.save(CHECKPOINT_DIR / f'ids_{experiment_name}_5fold.npy',   ids_ref)
        save_submission(ids_ref, mean_probs,
                        SUBMISSION_DIR / f'submission_{experiment_name}_5fold_tta_labels0.csv')
    else:
        print(f'Pas de CSV 5-fold final : {len(fold_probs)}/{N_SPLITS} folds OK')

    display(results_df)
    if not results_df.empty:
        print(f"Mean val acc: {results_df['best_val_acc'].mean():.4f} "
              f"+/- {results_df['best_val_acc'].std():.4f}")
    return results_df

print('run_5fold_patch OK')


run_5fold_patch OK


## 11. Run A — Patch 384×384 (300 epochs, seed=123)

In [11]:
patch_results = run_5fold_patch(
    experiment_name = 'seresnet50_bilinear_v3_patch',
    model_factory   = seresnet50_bilinear_v3,
    start_fold      = START_FOLD_PATCH,
)



seresnet50_bilinear_v3_patch | fold 1/5


  0%|          | 0/59 [00:00<?, ?it/s]

  0%|          | 0/15 [00:00<?, ?it/s]

seresnet50_bilinear_v3_patch_fold1 | ep 001 | train 0.1229/2.0800 | val 0.1332/2.0785 | best 0.1332 @ 1


  0%|          | 0/59 [00:00<?, ?it/s]

...

SWA (106 snapshots) val acc: 0.7733  |  best ckpt: 0.7797
-> Keeping best checkpoint


  0%|          | 0/25 [00:00<?, ?it/s]

Saved: /home/onyxia/work/tilda-texture-classification/outputs/submissions/submission_seresnet50_bilinear_v3_patch_fold5_tta_labels0.csv
label
0    183
1     61
2     78
3     89
4     82
5     89
6     98
7    109
Name: count, dtype: int64
Saved: /home/onyxia/work/tilda-texture-classification/outputs/submissions/submission_seresnet50_bilinear_v3_patch_5fold_tta_labels0.csv
label
0    161
1     86
2    111
3     97
4     56
5     88
6    104
7     86
Name: count, dtype: int64


,fold,best_val_acc,best_epoch,training_time_min
0,1,0.845666,295,200.337123
1,2,0.758475,268,200.645059
2,3,0.796610,286,200.685333
3,4,0.764831,270,200.667881
4,5,0.779661,287,200.320507


Mean val acc: 0.7890 +/- 0.0349


## 12. Ensemble final — v1 + v2_patch + v2_full + v3

In [12]:
# Ensemble v3 + v1 + v2 (4 modeles)
v1_patch = np.load(CHECKPOINT_DIR / 'probs_seresnet50_bilinear_patch_v1_5fold.npy')
v2_patch = np.load(CHECKPOINT_DIR / 'probs_seresnet50_bilinear_v2_patch_5fold.npy')
v2_full  = np.load(CHECKPOINT_DIR / 'probs_seresnet50_bilinear_v2_full_5fold.npy')
v3_patch = np.load(CHECKPOINT_DIR / 'probs_seresnet50_bilinear_v3_patch_5fold.npy')
ids_ref  = np.load(CHECKPOINT_DIR / 'ids_seresnet50_bilinear_patch_v1_5fold.npy')

# Mega ensemble 4 modeles : poids egaux
mega4 = (v1_patch + v2_patch + v2_full + v3_patch) / 4.0
save_submission(ids_ref, mega4,
    SUBMISSION_DIR / 'submission_mega4_v1v2v3_labels0.csv')

# Avec v3 plus pondere (modele le plus recent)
mega4_v3boost = 0.20 * v1_patch + 0.20 * v2_patch + 0.20 * v2_full + 0.40 * v3_patch
save_submission(ids_ref, mega4_v3boost,
    SUBMISSION_DIR / 'submission_mega4_v3boost_labels0.csv')


Saved: /home/onyxia/work/tilda-texture-classification/outputs/submissions/submission_mega4_v1v2v3_labels0.csv
label
0    132
1     89
2    100
3     97
4     89
5     91
6     98
7     93
Name: count, dtype: int64
Saved: /home/onyxia/work/tilda-texture-classification/outputs/submissions/submission_mega4_v3boost_labels0.csv
label
0    137
1     89
2    103
3     96
4     86
5     91
6     98
7     89
Name: count, dtype: int64


,id,label
0,1,3
1,2,4
2,3,4
3,4,0
4,5,2
...,...,...
784,785,1
785,786,0
786,787,7
787,788,2
